# TrendWaveletAE Pure-Stack Study Analysis

Compares pure homogeneous stacks of TrendWaveletAE and TrendWaveletAELG blocks
across datasets, and against the alternating TrendAE + WaveletV3AE study results.

**Key scientific question**: Does embedding trend + wavelet in one block (pure stack)
perform as well as or better than separating them across two alternating block types?

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 100)

ROOT = Path('experiments')
if not ROOT.exists():
    ROOT = Path('.')

RESULTS = ROOT / 'results'
REPORT_PATH = ROOT / 'analysis_reports' / 'trendwaveletae_pure_study_analysis.md'

DATASET_ORDER = ['m4', 'tourism', 'traffic', 'weather']
DATASET_LABELS = {
    'm4': 'M4-Yearly',
    'tourism': 'Tourism-Yearly',
    'traffic': 'Traffic-96',
    'weather': 'Weather-96',
}

STUDY_VARIANTS = {
    'ae': {
        'label': 'TrendWaveletAE',
        'search_csvs': {ds: RESULTS / ds / 'trendwaveletae_pure_study_results.csv' for ds in DATASET_ORDER},
        'cross_csv': RESULTS / 'trendwaveletae_pure_cross_dataset_results.csv',
    },
    'aelg': {
        'label': 'TrendWaveletAELG',
        'search_csvs': {ds: RESULTS / ds / 'trendwaveletaelg_pure_study_results.csv' for ds in DATASET_ORDER},
        'cross_csv': RESULTS / 'trendwaveletaelg_pure_cross_dataset_results.csv',
    },
}

# Alternating study results for comparison
ALTERNATING_CSVS = {
    ds: RESULTS / ds / 'wavelet_v3ae_study_results.csv' for ds in DATASET_ORDER
}

In [2]:
def load_csv(path: Path):
    if not path.exists():
        return None
    df = pd.read_csv(path)
    for c in ['search_round', 'best_val_loss', 'trend_dim_cfg', 'trend_thetas_dim_cfg',
              'latent_dim_cfg', 'basis_dim']:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    return df

# Load all data
all_data = {}
for vkey, vinfo in STUDY_VARIANTS.items():
    search_dfs = {ds: load_csv(path) for ds, path in vinfo['search_csvs'].items()}
    cross_df = load_csv(vinfo['cross_csv'])
    all_data[vkey] = {'search_dfs': search_dfs, 'cross_df': cross_df, 'label': vinfo['label']}

    print(f'--- {vinfo["label"]} ---')
    for ds in DATASET_ORDER:
        df = search_dfs[ds]
        if df is None:
            print(f'  {ds}: missing')
        else:
            rounds = sorted(df['search_round'].dropna().astype(int).unique().tolist())
            print(f'  {ds}: rows={len(df)} configs={df["config_name"].nunique()} rounds={rounds}')
    print(f'  cross rows: {0 if cross_df is None else len(cross_df)}')

# Load alternating study data
alternating_dfs = {ds: load_csv(path) for ds, path in ALTERNATING_CSVS.items()}
print('\n--- Alternating (WaveletV3AE) ---')
for ds in DATASET_ORDER:
    df = alternating_dfs[ds]
    if df is None:
        print(f'  {ds}: missing')
    else:
        rounds = sorted(df['search_round'].dropna().astype(int).unique().tolist())
        print(f'  {ds}: rows={len(df)} configs={df["config_name"].nunique()} rounds={rounds}')

--- TrendWaveletAE ---
  m4: rows=2133 configs=336 rounds=[1, 2, 3]
  tourism: missing
  traffic: missing
  weather: missing
  cross rows: 0
--- TrendWaveletAELG ---
  m4: rows=711 configs=112 rounds=[1, 2, 3]
  tourism: missing
  traffic: missing
  weather: missing
  cross rows: 0

--- Alternating (WaveletV3AE) ---
  m4: rows=1161 configs=332 rounds=[1]
  tourism: missing
  traffic: missing
  weather: missing


## Section 1: Per-Dataset Search Results

Round-by-round survival, convergence rates, top-10 leaderboards.

In [3]:
report = []
report.append('# TrendWaveletAE Pure-Stack Study Analysis')
report.append('')

for vkey, vdata in all_data.items():
    label = vdata['label']
    search_dfs = vdata['search_dfs']
    report.append(f'# Block Type: {label}')
    report.append('')

    for ds in DATASET_ORDER:
        df = search_dfs[ds]
        report.append(f'## {label} — Dataset: {DATASET_LABELS[ds]}')
        report.append('')
        if df is None or df.empty:
            report.append(f'- Missing CSV: `{STUDY_VARIANTS[vkey]["search_csvs"][ds]}`')
            report.append('')
            continue

        report.append(f'- Rows: {len(df)}')
        report.append(f'- Unique configs: {df["config_name"].nunique()}')
        report.append('')

        # Successive-halving funnel
        report.append('### Successive-Halving Funnel')
        rounds = sorted(df['search_round'].dropna().astype(int).unique().tolist())
        funnel_rows = []
        prev = None
        for r in rounds:
            rdf = df[df['search_round'].astype('Int64') == r]
            n_cfg = rdf['config_name'].nunique()
            keep = '-' if prev is None else f'{n_cfg}/{prev} ({n_cfg/max(prev,1):.0%})'
            best = pd.to_numeric(rdf['best_val_loss'], errors='coerce').median()
            funnel_rows.append({'Round': r, 'Configs': n_cfg, 'Rows': len(rdf),
                                'Median best_val_loss': best, 'Kept': keep})
            prev = n_cfg
        report.append(pd.DataFrame(funnel_rows).to_markdown(index=False, floatfmt='.4f'))
        report.append('')

        # Round 3 leaderboard
        report.append('### Round 3 Leaderboard (Top 10)')
        r3 = df[df['search_round'].astype('Int64') == 3].copy()
        if r3.empty:
            report.append('Round 3 not available.')
            report.append('')
        else:
            td_col = 'trend_dim_cfg' if 'trend_dim_cfg' in r3.columns else 'trend_thetas_dim_cfg'
            grp = (
                r3.groupby(['config_name', 'wavelet_type', 'bd_label', td_col, 'latent_dim_cfg'], dropna=False)
                .agg(median_best_val_loss=('best_val_loss', 'median'),
                     std_best_val_loss=('best_val_loss', 'std'),
                     n=('best_val_loss', 'count'))
                .sort_values('median_best_val_loss')
                .head(10)
                .reset_index()
            )
            report.append(grp.to_markdown(index=False, floatfmt='.4f'))
            report.append('')

print('\n'.join(report[:60]))

# TrendWaveletAE Pure-Stack Study Analysis

# Block Type: TrendWaveletAE

## TrendWaveletAE — Dataset: M4-Yearly

- Rows: 2133
- Unique configs: 336

### Successive-Halving Funnel
|   Round |   Configs |   Rows |   Median best_val_loss | Kept          |
|--------:|----------:|-------:|-----------------------:|:--------------|
|       1 |       336 |   1008 |                16.2828 | -             |
|       2 |       225 |    675 |                14.9406 | 225/336 (67%) |
|       3 |       150 |    450 |                13.7287 | 150/225 (67%) |

### Round 3 Leaderboard (Top 10)
| config_name                           | wavelet_type   | bd_label   |   trend_dim_cfg |   latent_dim_cfg |   median_best_val_loss |   std_best_val_loss |   n |
|:--------------------------------------|:---------------|:-----------|----------------:|-----------------:|-----------------------:|--------------------:|----:|
| TrendWaveletAE_sym10_eq_fcast_td3_ld8 | sym10          | eq_fcast   |               3 |   

## Section 2: AE vs LG Comparison

Paired metrics across matching hyperparameter settings.

In [4]:
report.append('# AE vs LG Comparison')
report.append('')

for ds in DATASET_ORDER:
    report.append(f'## {DATASET_LABELS[ds]}')
    report.append('')

    ae_df = all_data.get('ae', {}).get('search_dfs', {}).get(ds)
    lg_df = all_data.get('aelg', {}).get('search_dfs', {}).get(ds)

    if ae_df is None or lg_df is None:
        report.append('Not enough data (need both AE and LG results).')
        report.append('')
        continue

    # Use round 3 if available, else latest round
    for label, df in [('AE', ae_df), ('LG', lg_df)]:
        max_round = df['search_round'].dropna().max()
        report.append(f'- {label}: latest round = {int(max_round) if pd.notna(max_round) else "N/A"}')

    ae_r3 = ae_df[ae_df['search_round'].astype('Int64') == 3].copy()
    lg_r3 = lg_df[lg_df['search_round'].astype('Int64') == 3].copy()

    if ae_r3.empty or lg_r3.empty:
        report.append('Round 3 not available for both variants.')
        report.append('')
        continue

    # Compute median by wavelet_type to compare
    ae_med = ae_r3.groupby('wavelet_type')['best_val_loss'].median().rename('AE')
    lg_med = lg_r3.groupby('wavelet_type')['best_val_loss'].median().rename('LG')
    cmp = pd.concat([ae_med, lg_med], axis=1).dropna()
    cmp['diff'] = cmp['AE'] - cmp['LG']
    cmp['winner'] = cmp.apply(lambda r: 'LG' if r['diff'] > 0 else ('AE' if r['diff'] < 0 else 'tie'), axis=1)

    report.append(cmp.sort_values('diff').to_markdown(floatfmt='.4f'))
    report.append('')

    wins = cmp['winner'].value_counts()
    report.append(f'Win/Loss/Draw: AE={wins.get("AE", 0)}, LG={wins.get("LG", 0)}, tie={wins.get("tie", 0)}')

    # Wilcoxon signed-rank test
    if len(cmp) >= 5:
        stat, pval = stats.wilcoxon(cmp['AE'], cmp['LG'])
        report.append(f'Wilcoxon signed-rank test: stat={stat:.4f}, p={pval:.4f}')
    report.append('')

print('--- AE vs LG comparison done ---')

--- AE vs LG comparison done ---


## Section 3: Pure vs Alternating Comparison

The main result. Compare pure-stack TrendWaveletAE configs to alternating TrendAE + WaveletV3AE.

In [5]:
report.append('# Pure vs Alternating Comparison')
report.append('')

# Map wavelet_type string to WaveletV3AE block name
WAVELET_TYPE_TO_BLOCK = {
    'haar': 'HaarWaveletV3AE', 'db2': 'DB2WaveletV3AE', 'db3': 'DB3WaveletV3AE',
    'db4': 'DB4WaveletV3AE', 'db10': 'DB10WaveletV3AE', 'db20': 'DB20WaveletV3AE',
    'coif1': 'Coif1WaveletV3AE', 'coif2': 'Coif2WaveletV3AE', 'coif3': 'Coif3WaveletV3AE',
    'coif10': 'Coif10WaveletV3AE', 'sym2': 'Symlet2WaveletV3AE', 'sym3': 'Symlet3WaveletV3AE',
    'sym10': 'Symlet10WaveletV3AE', 'sym20': 'Symlet20WaveletV3AE',
}

for ds in DATASET_ORDER:
    report.append(f'## {DATASET_LABELS[ds]}')
    report.append('')

    pure_df = all_data.get('ae', {}).get('search_dfs', {}).get(ds)
    alt_df = alternating_dfs.get(ds)

    if pure_df is None or alt_df is None:
        report.append('Not enough data for comparison.')
        report.append('')
        continue

    pure_r3 = pure_df[pure_df['search_round'].astype('Int64') == 3].copy()
    alt_r3 = alt_df[alt_df['search_round'].astype('Int64') == 3].copy()

    if pure_r3.empty or alt_r3.empty:
        report.append('Round 3 not available for comparison.')
        report.append('')
        continue

    # Overall comparison
    pure_med = pure_r3['best_val_loss'].median()
    alt_med = alt_r3['best_val_loss'].median()
    report.append(f'- Pure median val_loss: {pure_med:.4f}')
    report.append(f'- Alternating median val_loss: {alt_med:.4f}')
    report.append(f'- Difference (pure - alt): {pure_med - alt_med:.4f}')
    report.append('')

    # By wavelet family
    report.append('### By Wavelet Family')
    pure_by_wf = pure_r3.groupby('wavelet_type')['best_val_loss'].median().rename('pure')
    alt_r3_wf = alt_r3.copy()
    if 'wavelet_family' in alt_r3_wf.columns:
        alt_by_wf = alt_r3_wf.groupby('wavelet_family')['best_val_loss'].median()
        # Remap alternating block names to wavelet_type strings
        block_to_type = {v: k for k, v in WAVELET_TYPE_TO_BLOCK.items()}
        alt_by_wf.index = alt_by_wf.index.map(lambda x: block_to_type.get(x, x))
        alt_by_wf = alt_by_wf.rename('alternating')
        cmp = pd.concat([pure_by_wf, alt_by_wf], axis=1).dropna()
        cmp['diff'] = cmp['pure'] - cmp['alternating']
        cmp = cmp.sort_values('diff')
        report.append(cmp.to_markdown(floatfmt='.4f'))
        report.append('')

        wins = (cmp['diff'] < 0).sum()
        losses = (cmp['diff'] > 0).sum()
        ties = (cmp['diff'] == 0).sum()
        report.append(f'Pure wins: {wins}, Alternating wins: {losses}, Ties: {ties}')
        report.append('')

print('--- Pure vs Alternating comparison done ---')

--- Pure vs Alternating comparison done ---


## Section 4: Cross-Dataset Results

Global top-10 and dataset-specific rankings.

In [6]:
report.append('# Cross-Dataset Results')
report.append('')

for vkey, vdata in all_data.items():
    label = vdata['label']
    cross_df = vdata['cross_df']
    report.append(f'## {label}')
    report.append('')
    if cross_df is None or cross_df.empty:
        report.append(f'- Missing or empty cross CSV: `{STUDY_VARIANTS[vkey]["cross_csv"]}`')
        report.append('')
        continue

    metric = 'best_val_loss'
    if metric not in cross_df.columns and 'final_val_loss' in cross_df.columns:
        metric = 'final_val_loss'

    board = (
        cross_df.groupby(['canonical_config_id', 'source_datasets'], dropna=False)
        .agg(mean_metric=(metric, 'mean'), std_metric=(metric, 'std'), n=(metric, 'count'))
        .sort_values('mean_metric')
        .reset_index()
    )
    report.append(board.to_markdown(index=False, floatfmt='.4f'))
    report.append('')
    report.append(f'- Total cross rows: {len(cross_df)}')
    report.append('')

print('--- Cross-dataset done ---')

--- Cross-dataset done ---


## Section 5: Hyperparameter Sensitivity

Marginal medians by wavelet family, basis label, trend_dim, latent_dim.

In [7]:
report.append('# Hyperparameter Sensitivity')
report.append('')

for vkey, vdata in all_data.items():
    label = vdata['label']
    report.append(f'## {label}')
    report.append('')

    for ds in DATASET_ORDER:
        df = vdata['search_dfs'][ds]
        report.append(f'### {DATASET_LABELS[ds]}')
        report.append('')
        if df is None or df.empty:
            report.append('No data.')
            report.append('')
            continue

        r1 = df[df['search_round'].astype('Int64') == 1].copy()
        if r1.empty:
            report.append('No round-1 data.')
            report.append('')
            continue

        for col in ['wavelet_type', 'wavelet_family', 'bd_label', 'trend_dim_cfg', 'latent_dim_cfg']:
            if col not in r1.columns:
                continue
            mg = (
                r1.groupby(col)
                .agg(median_best_val_loss=('best_val_loss', 'median'),
                     mean_best_val_loss=('best_val_loss', 'mean'),
                     n=('best_val_loss', 'count'))
                .sort_values('median_best_val_loss')
                .reset_index()
            )
            report.append(f'#### {col}')
            report.append(mg.to_markdown(index=False, floatfmt='.4f'))
            report.append('')

print('--- Hyperparameter sensitivity done ---')

--- Hyperparameter sensitivity done ---


In [8]:
# Write final report
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text('\n'.join(report), encoding='utf-8')
print(f'Wrote report to: {REPORT_PATH}')
print(f'Total lines: {len(report)}')
print('\n'.join(report[:30]))

Wrote report to: experiments/analysis_reports/trendwaveletae_pure_study_analysis.md
Total lines: 167
# TrendWaveletAE Pure-Stack Study Analysis

# Block Type: TrendWaveletAE

## TrendWaveletAE — Dataset: M4-Yearly

- Rows: 2133
- Unique configs: 336

### Successive-Halving Funnel
|   Round |   Configs |   Rows |   Median best_val_loss | Kept          |
|--------:|----------:|-------:|-----------------------:|:--------------|
|       1 |       336 |   1008 |                16.2828 | -             |
|       2 |       225 |    675 |                14.9406 | 225/336 (67%) |
|       3 |       150 |    450 |                13.7287 | 150/225 (67%) |

### Round 3 Leaderboard (Top 10)
| config_name                           | wavelet_type   | bd_label   |   trend_dim_cfg |   latent_dim_cfg |   median_best_val_loss |   std_best_val_loss |   n |
|:--------------------------------------|:---------------|:-----------|----------------:|-----------------:|-----------------------:|--------------------